In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

#데이터 불러오기 
df = pd.read_csv('./model_df_new_cat_음수할인율처리후.csv')
print(df.columns)
df.head(1)


Index(['기획년도', '주차', '카테고리_통합', '시즌이월', '시즌', '총입고수량', '판매수량', '판매액', '평균택가',
       '평균원가', '주차별_평균_실판매가', '월별_평균_실판매가', '시즌별_평균_실판매가', '총입고원가', '총입고택가',
       '매출원가계', '판매택가계', '실판매가', '할인율', '누적판매수량', '누적판매액', '누적매출원가', '누적판매택가',
       '누적판매율', 'ROI', '악천후일수', '평균기온(도)'],
      dtype='object')


,기획년도,주차,카테고리_통합,시즌이월,시즌,총입고수량,판매수량,판매액,평균택가,평균원가,...,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,악천후일수,평균기온(도)
0,2021,2021-07-11,가을_가성비 수트셋업,01_시즌,가을,13163,496,84636400,184000,28105,...,170638,7,496,84636400,13940080,91264000,3.77,0.17,0,28.5


In [ ]:
# 피벗테이블로 4년동안 데이터가 있는 카테고리만 남김
# 카테고리별 연도 존재 여부 확인 (0: 없음, 1: 있음)
category_year_table = df.groupby(["카테고리_통합", "기획년도"]).size().unstack(fill_value=0)
# 연도가 존재하면 1로 변환 (카테고리가 존재했음을 의미)
category_year_table = (category_year_table > 0).astype(int)

# 21, 22, 23, 24년도 모두 존재한 카테고리만 필터링
categories_all_years = category_year_table[
    (category_year_table.get(2021, 0) == 1) & 
    (category_year_table.get(2022, 0) == 1) & 
    (category_year_table.get(2023, 0) == 1) & 
    (category_year_table.get(2024, 0) == 1)
].index

# 새로운 데이터프레임 생성
df = df[df["카테고리_통합"].isin(categories_all_years)].copy()

In [ ]:
# 2023년도 데이터(지난 1년동안의 데이터)로 카테고리 선정하고자 함
df = df[df['기획년도']==2023]

In [112]:
# 기준 1) 매출 기여도가 높은 핵심 카테고리 (판매량 & 매출 기준)
top_sales = df.groupby("카테고리_통합", group_keys=False).agg(
    총판매수량=("판매수량", "sum"),
    총매출=("판매액", "sum"),
    총입고수량=("총입고수량", "max")
).reset_index()

# 기준 2) 재고 부담 & 할인 전략 개선이 필요한 카테고리 (최대 누적판매율 & 평균 할인율 & 시즌 종료 시 할인율 상승 패턴 확인)
top_inventory_adjusted = df.groupby("카테고리_통합", group_keys=False).agg(
    최대누적판매율=("누적판매율", "max"),  # 같은 연도 내 최대 누적판매율
    평균할인율=("할인율", "mean")  # 평균 할인율 유지
).reset_index()

# 시즌 종료에 따라 할인율이 급격히 증가하는 패턴 확인
df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U") #24-32주차 이런식으로 나옴

# 각 카테고리별 주차별 할인율 변화 측정
discount_trend = df.groupby(["카테고리_통합", "연도주차"], group_keys=False)["할인율"].mean().reset_index()
discount_trend["할인율_변화"] = discount_trend.groupby("카테고리_통합", group_keys=False)["할인율"].diff() #직전 주차에 비해 얼마나 할인율이 상승했는지

# 마지막 4주 동안 할인율이 과하게 상승한 카테고리 찾기
last_weeks = discount_trend.groupby("카테고리_통합", group_keys=False).tail(4)  # 마지막 4주 데이터
discount_rise = last_weeks.groupby("카테고리_통합", group_keys=False)["할인율_변화"].sum().reset_index()
discount_rise.columns = ["카테고리_통합", "시즌 종료 할인율 증가량"]

# 마지막 주차의 할인율 추가
last_week_discount = df.groupby("카테고리_통합", group_keys=False).apply(
    lambda x: x.loc[x["주차"] == x["주차"].max(), "할인율"].mean()
).reset_index()
last_week_discount.columns = ["카테고리_통합", "마지막 주차 할인율"]

# 기준 3) 가격 탄력성이 높은 카테고리 (할인율-판매량 상관관계)
top_correlation = df.groupby("카테고리_통합", group_keys=False).apply(
    lambda x: x["할인율"].corr(x["판매수량"])
).reset_index()
top_correlation.columns = ["카테고리_통합", "가격 탄력성 (할인율-판매량 상관관계)"]

# 모든 데이터 병합
category_selection = (
    top_sales
    .merge(top_inventory_adjusted, on="카테고리_통합", how="left")
    .merge(last_week_discount, on="카테고리_통합", how="left")
    .merge(top_correlation, on="카테고리_통합", how="left")
    .merge(discount_rise, on="카테고리_통합", how="left")
    .fillna(0)  # NaN 값은 0으로 채움
)

# 소수점 라운딩
category_selection["평균할인율"] = category_selection["평균할인율"].round(2)
category_selection["가격 탄력성 (할인율-판매량 상관관계)"] = category_selection["가격 탄력성 (할인율-판매량 상관관계)"].round(2)
category_selection["마지막 주차 할인율"] = category_selection["마지막 주차 할인율"].astype(int)

# 정렬 및 순위 계산 (최대 누적판매율은 낮을수록 우선순위 → 오름차순, 나머지는 높을수록 우선순위 → 내림차순)
category_selection["총매출_순위"] = category_selection["총매출"].rank(method="min", ascending=False).astype(int)
category_selection["총입고_순위"] = category_selection["총입고수량"].rank(method="min", ascending=False).astype(int)
category_selection["최대누적판매율_순위"] = category_selection["최대누적판매율"].rank(method="min", ascending=True).astype(int)  # 낮을수록 재고 부담 ↑
category_selection["평균할인율_순위"] = category_selection["평균할인율"].rank(method="min", ascending=False).astype(int)
category_selection["시즌 종료 할인율 증가량_순위"] = category_selection["시즌 종료 할인율 증가량"].rank(method="min", ascending=False).astype(int)
category_selection["가격 탄력성_순위"] = category_selection["가격 탄력성 (할인율-판매량 상관관계)"].rank(method="min", ascending=False).astype(int)

# 각 기준별 점수 계산
# 기준 1: 매출 기여도 높은 카테고리 (총매출, 총입고수량)
category_selection["기준1_점수"] = category_selection["총매출_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["총입고_순위"].rank(method="min", ascending=True).astype(int)

# 기준 2: 재고 부담 & 할인 전략 개선 (최대누적판매율, 평균할인율, 시즌 종료 할인율 증가량)
category_selection["기준2_점수"] = category_selection["최대누적판매율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["평균할인율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["시즌 종료 할인율 증가량_순위"].rank(method="min", ascending=True).astype(int)

# 기준 3: 가격 탄력성이 높은 카테고리
category_selection["기준3_점수"] = category_selection["가격 탄력성_순위"].rank(method="min", ascending=True).astype(int)

# 최종 점수 계산 (기준 1, 2, 3 점수를 모두 합산)
category_selection["최종_점수"] = category_selection["기준1_점수"] + category_selection["기준2_점수"] + category_selection["기준3_점수"]

# 기준 점수를 기준으로 정렬
# category_selection_sorted = category_selection.sort_values(by=["기준1_점수"], ascending=True)
# category_selection_sorted = category_selection.sort_values(by=["기준2_점수"], ascending=True)
category_selection_sorted = category_selection.sort_values(by=["기준3_점수"], ascending=True)
# category_selection_sorted = category_selection.sort_values(by=["최종_점수"], ascending=True)
category_selection_sorted


,카테고리_통합,총판매수량,총매출,총입고수량,최대누적판매율,평균할인율,마지막 주차 할인율,가격 탄력성 (할인율-판매량 상관관계),시즌 종료 할인율 증가량,총매출_순위,총입고_순위,최대누적판매율_순위,평균할인율_순위,시즌 종료 할인율 증가량_순위,가격 탄력성_순위,기준1_점수,기준2_점수,기준3_점수,최종_점수
13,겨울_해비스웨터,16262,1089973951,37081,43.86,39.00,61,0.96,9.0,20,14,8,25,1,1,34,34,1,69
7,겨울_기모맨투맨셋업,19472,782652236,47829,40.71,51.52,69,0.92,0.0,22,12,4,12,15,2,34,31,2,67
5,겨울_가성비 수트셋업,25674,3875276475,49137,52.25,31.90,37,0.88,0.0,3,11,17,29,15,3,14,61,3,78
12,겨울_팬츠,30167,1787226962,66847,45.13,36.00,53,0.88,3.0,16,8,11,26,10,3,24,47,3,74
11,겨울_코트,18621,3864696795,42273,44.05,48.62,63,0.86,1.0,4,13,9,15,13,5,17,37,5,59
20,사계절_가성비 수트셋업,76589,8830897666,120246,63.69,39.36,48,0.74,-7.0,1,4,24,22,30,6,5,76,6,87
2,가을_캐주얼셔츠,6249,252576620,11824,52.85,41.48,72,0.73,8.0,31,30,18,21,4,7,61,43,7,111
8,겨울_스웨터,22361,1170991922,53911,41.48,43.20,61,0.72,8.0,19,10,5,19,4,8,29,28,8,65
31,여름_캐주얼셔츠,97697,2959959897,158857,61.50,62.00,76,0.67,-2.0,8,1,23,4,21,9,9,48,9,66
32,여름_팬츠,78720,2886590147,118291,66.55,57.31,73,0.67,-1.0,12,5,27,10,19,9,17,56,9,82
